In [ ]:
import fitz  # PyMuPDF
import edge_tts
import asyncio
import re
from deep_translator import GoogleTranslator

PDF_FILE = "libro.pdf"

# Configuración de voces neuronales
VOICE_EN = "en-US-GuyNeural"      # Opciones: en-US-JennyNeural, en-GB-RyanNeural
VOICE_ES = "es-ES-AlvaroNeural"   # Opciones: es-MX-DaliaNeural, es-ES-ElviraNeural

# Modos de exportación: 'en', 'es', o 'both'
EXPORT_MODE = "both"

def clean_text(text: str) -> str:
    """Elimina cortes de palabras por guion al final de línea y normaliza espacios."""
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)  # Une palabras cortadas: "exam-\nple" -> "example"
    text = re.sub(r'\s+', ' ', text)                 # Reemplaza múltiples espacios y saltos de línea
    return text.strip()

def translate_to_spanish(text: str) -> str:
    """Traduce fragmentos largos respetando los límites de la API de Google."""
    translator = GoogleTranslator(source='en', target='es')
    # GoogleTranslator maneja internamente bloques grandes, pero dividimos si excede ~4000 caracteres
    chunks = [text[i:i + 4000] for i in range(0, len(text), 4000)]
    translated_chunks = [translator.translate(chunk) for chunk in chunks]
    return " ".join(translated_chunks)

async def synthesize(text: str, voice: str, filename: str):
    """Genera el archivo de audio usando edge-tts."""
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(filename)

async def process_pdf():
    doc = fitz.open(PDF_FILE)
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        raw_text = page.get_text()
        text_en = clean_text(raw_text)
        
        # Ignorar páginas en blanco o portadas sin texto legible
        if not text_en or len(text_en) < 30:
            continue
            
        page_id = f"{page_num + 1:03d}"
        print(f"\n--- Procesando Página {page_id} ---")
        
        # 1. Pista en Inglés (Original)
        if EXPORT_MODE in ("en", "both"):
            filename_en = f"audio_en_pagina_{page_id}.mp3"
            print(f"Generando audio inglés: {filename_en}...")
            await synthesize(text_en, VOICE_EN, filename_en)
            
        # 2. Pista en Español (Traducida)
        if EXPORT_MODE in ("es", "both"):
            filename_es = f"audio_es_pagina_{page_id}.mp3"
            print(f"Traduciendo página {page_id} al español...")
            text_es = translate_to_spanish(text_en)
            
            print(f"Generando audio español: {filename_es}...")
            await synthesize(text_es, VOICE_ES, filename_es)

if __name__ == "__main__":
    asyncio.run(process_pdf())